# Ch.6 — Gradient + Matrix Chain Rule

**Running theme.** Tune all eight free-throw parameters at once. Scalar derivative → vector gradient. Scalar chain rule → matrix chain rule. Two upgrades, and we have backpropagation.

**Learning outcomes.**
1. Compute gradients on a 2-D loss surface and watch descent from any starting point.
2. Track Jacobian shapes through a small neural-network layer.
3. Verify analytical gradients against finite differences and against PyTorch autograd.
4. See why reverse-mode autodiff (backprop) beats forward-mode when there are many parameters and one scalar loss.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, FloatText, FloatLogSlider, IntSlider, HBox, VBox, Output, jslink

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (7.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})
np.set_printoptions(precision=4, suppress=True)

def linked_pair(label, value, vmin, vmax, step=0.05):
    sl = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                     description=label, continuous_update=True, readout=False)
    tx = FloatText(value=value, description=" ", step=step,
                   layout={"width": "110px"})
    jslink((sl, "value"), (tx, "value"))
    return sl, tx

## 2 · The gradient points uphill — verify numerically

For $f(\theta_1,\theta_2) = \tfrac{1}{2}(\theta_1^2 + 3\theta_2^2 + 1.2\,\theta_1\theta_2)$, compute $\nabla f$ analytically, then confirm by finite differences.

In [ ]:
def f(theta):
    t1, t2 = theta
    return 0.5 * (t1**2 + 3*t2**2 + 1.2*t1*t2)

def grad_f(theta):
    t1, t2 = theta
    return np.array([t1 + 0.6*t2, 3*t2 + 0.6*t1])

theta = np.array([1.3, -0.8])
g_analytic = grad_f(theta)

# Finite-difference gradient
eps = 1e-6
g_numeric = np.array([
    (f(theta + eps*np.array([1, 0])) - f(theta - eps*np.array([1, 0]))) / (2*eps),
    (f(theta + eps*np.array([0, 1])) - f(theta - eps*np.array([0, 1]))) / (2*eps),
])

print(f"analytic  ∇f = {g_analytic}")
print(f"numeric   ∇f = {g_numeric}")
print(f"relative err = {np.linalg.norm(g_analytic - g_numeric)/np.linalg.norm(g_analytic):.2e}")

## 3 · Interactive — choose the starting point and step size, watch the trajectory

Drag the $(\theta_1^{(0)}, \theta_2^{(0)})$ sliders to pick a starting point. Drag $\eta$ to change step size. Too big → zigzag along the narrow valley. Too small → creep.

In [ ]:
x1_sl, x1_tx = linked_pair("θ1(0)", value=-2.0, vmin=-2.4, vmax=2.4, step=0.05)
x2_sl, x2_tx = linked_pair("θ2(0)",  value=1.6, vmin=-1.9, vmax=1.9, step=0.05)
eta_sl = FloatLogSlider(value=0.18, base=10, min=-2.3, max=-0.1, step=0.05,
                        description="η", continuous_update=True)
iters_sl = IntSlider(value=30, min=3, max=80, step=1, description="iters",
                     continuous_update=True)
readout = Output()

grid = np.linspace(-2.5, 2.5, 200)
X, Y = np.meshgrid(grid, np.linspace(-2, 2, 200))
Z = 0.5 * (X**2 + 3*Y**2 + 1.2*X*Y)

fig, ax = plt.subplots(figsize=(6.8, 5.2))
ax.contourf(X, Y, Z, levels=18, cmap="Blues_r", alpha=0.55)
ax.contour(X, Y, Z, levels=18, colors="#7F8C8D", linewidths=0.4)
ax.plot(0, 0, "*", color="#E74C3C", markersize=16, zorder=5)
(path_line,) = ax.plot([], [], "-o", color="#8E44AD", lw=1.8, markersize=4)
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2, 2); ax.set_aspect("equal")
ax.set_xlabel("θ1"); ax.set_ylabel("θ2")

def update_descent(*_):
    theta = np.array([x1_sl.value, x2_sl.value])
    eta = eta_sl.value
    path = [theta.copy()]
    for _ in range(iters_sl.value):
        theta = theta - eta * grad_f(theta)
        path.append(theta.copy())
        if np.linalg.norm(theta) > 20:
            break
    p = np.array(path)
    path_line.set_data(p[:, 0], p[:, 1])
    with readout:
        readout.clear_output(wait=True)
        print(f"steps taken      : {len(path) - 1}")
        print(f"final θ          : {p[-1]}")
        print(f"final f(θ)       : {f(p[-1]):.6f}")
        print(f"distance to optim: {np.linalg.norm(p[-1]):.4e}")
    fig.canvas.draw_idle()

for w in (x1_sl, x2_sl, eta_sl, iters_sl):
    w.observe(update_descent, names="value")
update_descent()

display(VBox([HBox([x1_sl, x1_tx, x2_sl, x2_tx]),
              HBox([eta_sl, iters_sl]),
              readout]))

## 4 · Jacobian shapes through one neural-network layer

Forward: $\mathbf{x} \to \mathbf{u} = W\mathbf{x} + \mathbf{b} \to \mathbf{h} = \sigma(\mathbf{u}) \to L$. Verify that each backward step has the Jacobian shape claimed in the README.

In [ ]:
rng = np.random.default_rng(42)
n, m = 5, 3                            # input dim, output dim of the layer

x = rng.normal(size=n)
W = rng.normal(size=(m, n)) * 0.5
b = rng.normal(size=m) * 0.1

def sigma(u):        return np.tanh(u)
def sigma_deriv(u):  return 1.0 - np.tanh(u) ** 2

# Forward pass
u = W @ x + b                              # (m,)
h = sigma(u)                               # (m,)
L = 0.5 * (h ** 2).sum()                   # scalar loss: ½ ||h||²

print(f"shapes:   x{x.shape},  W{W.shape},  u{u.shape},  h{h.shape},  L is scalar")

# Backward pass (right-to-left)
dL_dh = h                                  # (m,)   since L = ½||h||²
dL_du = dL_dh * sigma_deriv(u)             # (m,)   diag(σ')·dL/dh == elementwise
dL_dx = W.T @ dL_du                        # (n,)   pull back through W
dL_dW = np.outer(dL_du, x)                 # (m, n) weight gradient

print(f"dL/dh shape = {dL_dh.shape}   (should be {(m,)})")
print(f"dL/du shape = {dL_du.shape}   (should be {(m,)})")
print(f"dL/dx shape = {dL_dx.shape}   (should be {(n,)})")
print(f"dL/dW shape = {dL_dW.shape}   (should be {(m, n)})")

## 5 · Verify the layer gradients with finite differences

Analytical backprop is famously easy to typo. Always sanity-check against a numerical gradient on a small example.

In [ ]:
def loss_from_W(W_flat):
    Wm = W_flat.reshape(m, n)
    return 0.5 * (np.tanh(Wm @ x + b) ** 2).sum()

W_flat = W.flatten()
eps = 1e-6
grad_numeric = np.zeros_like(W_flat)
for i in range(len(W_flat)):
    plus  = W_flat.copy(); plus[i]  += eps
    minus = W_flat.copy(); minus[i] -= eps
    grad_numeric[i] = (loss_from_W(plus) - loss_from_W(minus)) / (2 * eps)
grad_numeric = grad_numeric.reshape(m, n)

err = np.linalg.norm(grad_numeric - dL_dW) / np.linalg.norm(dL_dW)
print(f"analytic dL/dW[0]  = {dL_dW[0]}")
print(f"numeric  dL/dW[0]  = {grad_numeric[0]}")
print(f"relative error     = {err:.2e}  (should be ≤ 1e-6)")

## 6 · Cross-check with PyTorch autograd (optional)

If PyTorch is installed, let the library compute the same gradient via reverse-mode autodiff and confirm the numbers match. If not, skip — the hand-derived backprop above is the ground truth.

In [ ]:
try:
    import torch
    W_t = torch.tensor(W, requires_grad=True)
    x_t = torch.tensor(x)
    b_t = torch.tensor(b)
    u_t = W_t @ x_t + b_t
    h_t = torch.tanh(u_t)
    L_t = 0.5 * (h_t ** 2).sum()
    L_t.backward()
    torch_grad = W_t.grad.numpy()
    err = np.linalg.norm(torch_grad - dL_dW) / np.linalg.norm(dL_dW)
    print(f"torch  dL/dW[0] = {torch_grad[0]}")
    print(f"hand   dL/dW[0] = {dL_dW[0]}")
    print(f"relative err    = {err:.2e}")
except ImportError:
    print("PyTorch not installed — skipping autograd cross-check.")
    print("The finite-difference check above already validates the hand derivation.")

## 7 · Forward-mode vs reverse-mode cost

For a stack of $L$ matrices of shape $d \times d$ applied to a $d$-vector input with a scalar output, forward mode costs $\mathcal{O}(L d^3)$ (computes full Jacobians). Reverse mode costs $\mathcal{O}(L d^2)$ (only matrix-vector products). Let's confirm empirically.

In [ ]:
import time
d, L_layers = 200, 12
Ws = [rng.normal(size=(d, d)) * (1/np.sqrt(d)) for _ in range(L_layers)]
x0 = rng.normal(size=d)

# Forward-mode: build the full Jacobian of the composite function.
t0 = time.perf_counter()
J = np.eye(d)
for Wl in Ws:
    J = Wl @ J                     # (d × d) × (d × d)
t_forward = time.perf_counter() - t0

# Reverse-mode: only propagate a gradient vector.
# Scalar loss L = sum(output).
t0 = time.perf_counter()
g = np.ones(d)                     # dL/d(output) for L = sum
for Wl in reversed(Ws):
    g = Wl.T @ g                   # (d × d) × (d)
t_reverse = time.perf_counter() - t0

print(f"forward-mode Jacobian build : {t_forward*1000:.2f} ms")
print(f"reverse-mode gradient       : {t_reverse*1000:.2f} ms")
print(f"speed-up                    : {t_forward/t_reverse:.1f}×")
print(f"\nreverse grad matches J.T @ 1s? {np.allclose(g, J.T @ np.ones(d))}")

## 8 · Where this reappears

- **ML Ch.5 Backprop & Optimisers.** The right-to-left multiplication in cell 4 is exactly what PyTorch's `.backward()` does, generalised to arbitrary computation graphs.
- **ML Ch.4 Neural Networks.** Stack the layer of cell 4 $L$ times.
- **ML Ch.18 Transformers.** Attention Jacobians are block-structured but obey the same chain rule.

**Final exercise.** Extend cell 4 to two layers: $\mathbf{x} \to \mathbf{h}_1 = \sigma(W_1 \mathbf{x}) \to \mathbf{h}_2 = \sigma(W_2 \mathbf{h}_1) \to L = \tfrac12 \|\mathbf{h}_2\|^2$. Write out $\nabla_{W_1} L$ and verify with finite differences. If you can do that, you can code backprop for any depth.